In [1]:
#import libraries

import scicone
import numpy as np
import pickle
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import io
import scanpy as sc
import anndata
import pyranges
import gseapy as gp

# Set up SCICoNE
install_path = '/cluster/work/bewi/members/andress/pylabs/SCICoNE/build/'
install_path_local = "/home/andress/pylabs/SCICoNE_lab/build/"
temporary_outpath = './'

seed = 42 # for reproducibility

np.random.seed(seed)

# Create SCICoNE object
sci = scicone.SCICoNE(install_path_local, temporary_outpath, verbose=False)

Using binaries at /home/andress/pylabs/SCICoNE_lab/build/


In [2]:


# Define paths
scdna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/cnv/'
scrna_path_ssh = '/cluster/work/bewi/members/andress/SCICoNE_lab/rna_imp/clonealign-processed-data/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19/'

scdna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/cnv'
scrna_path_local = '/home/andress/pylabs/SCICoNE_lab/rna_imp/SA501/10X/20171026_SA501X2XB00096/outs/filtered_gene_bc_matrices/hg19'

# Try SSH paths first
try:
    if os.path.exists(scdna_path_ssh) and os.path.exists(scrna_path_ssh):
        scdna_path = scdna_path_ssh
        scrna_path = scrna_path_ssh
    else:
        raise FileNotFoundError("SSH paths are not accessible.")
except FileNotFoundError:
    # Use local paths
    if os.path.exists(scdna_path_local) and os.path.exists(scrna_path_local):
        scdna_path = scdna_path_local
        scrna_path = scrna_path_local
    else:
        raise FileNotFoundError("Neither SSH nor local paths are accessible.")

In [3]:
adatas_path = '/home/andress/pylabs/SCICoNE_lab/rna_imp/adatas'


# Normalize and log transform with a cutoff to avoid negative values
for name in ['clusters_mean', 'clusters_sum', 'clusters_median']:
    cluster_data = anndata.read_h5ad(f'{adatas_path}/adata_{name}.h5ad')
    #sort the data by chromosome
    
    #sc.pp.normalize_total(cluster_data, target_sum=1e4)
    cluster_data.X = np.log1p(cluster_data.X + 1)
    cluster_data.write_h5ad(f'/home/andress/pylabs/SCICoNE_lab/rna_imp/{name}_transformed.h5ad')

    # Inspect transformed data
    print(f"Transformed {name}:", cluster_data.X[:5, :5])

    #print min and max values in transformed data
    print(f"Min and max values in {name}:", cluster_data.X.min(), cluster_data.X.max())



Transformed clusters_mean: [[0.69314718 0.69314718 0.69314718 0.71376647 0.71376647]
 [0.70967648 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.79112759 0.69314718 0.70774598 0.75030559 0.72213472]
 [0.74625701 0.70219702 0.70219702 0.70219702 0.70219702]
 [0.74893854 0.69314718 0.69314718 0.7094077  0.70131049]]
Min and max values in clusters_mean: 0.6931471805599453 6.189358664168741
Transformed clusters_sum: [[0.69314718 0.69314718 0.69314718 1.09861229 1.09861229]
 [1.09861229 0.69314718 0.69314718 0.69314718 0.69314718]
 [2.19722458 0.69314718 1.09861229 1.79175947 1.38629436]
 [2.07944154 1.09861229 1.09861229 1.09861229 1.09861229]
 [2.19722458 0.69314718 0.69314718 1.38629436 1.09861229]]
Min and max values in clusters_sum: 0.6931471805599453 10.067093391134142
Transformed clusters_median: [[0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0.69314718 0.69314718 0.69314718 0.69314718]
 [0.69314718 0